# Tutorial 9 — SSD and YOLO architectures and training

Implement the handout’s small prediction heads, verify their shapes, then train real pretrained SSD and YOLO models on the same public annotations. Shape checks alone are not object detection training.

**How to run:** select a Python kernel with the repository requirements, then Run All. QUICK mode keeps experiments short. For the larger run set `TUTORIAL_FULL=1` before starting the kernel. A quick run demonstrates the method; it is not a final accuracy benchmark.

**Execution status:** Complete implementation; end-to-end training/inference has not been verified in this repository run because the required large downloads did not finish. See `VALIDATION.md` for the separate offline model checks. Run All in Colab to generate your own results.

In [ ]:
# Google Colab: fetch the support files and install missing dependencies.
# Local Jupyter: follow README setup once; this cell makes no changes.
import sys, subprocess
from pathlib import Path
if "google.colab" in sys.modules:
    repo = Path("/content/dl-course-tutorials")
    if not repo.exists():
        subprocess.check_call(["git", "clone", "https://github.com/naveed952/Tutorial-01_Perceptron_py.ipynb.git", str(repo)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo/"requirements.txt")])
    import os
    os.chdir(repo)


## Step 1 — Locate the project and set a reproducible seed
A fixed seed makes comparisons easier to reproduce. Each notebook can run independently.

In [ ]:
from pathlib import Path
import sys, os
ROOT = Path.cwd() if (Path.cwd()/"tutorial_support").exists() else Path.cwd().parent
assert (ROOT/"tutorial_support").exists(), "Run from the repository or notebooks folder."
sys.path.insert(0, str(ROOT))
from tutorial_support.common import *
seed()
print("Mode:", "QUICK: small real-data run" if QUICK else "FULL: larger training run")

## Step 2 — Build small multi-scale SSD and grid YOLO heads
SSD has classification and box channels at each feature-map scale. The tutorial YOLO head predicts A(5+C) channels: four coordinates, objectness and class scores per anchor. This teaching head is not the internal architecture of modern YOLOv8.

In [ ]:
import torch
from torch import nn
DEVICE=torch_setup()
class TinySSD(nn.Module):
    def __init__(self,classes=3,anchors=3):
        super().__init__();self.blocks=nn.ModuleList([nn.Sequential(nn.Conv2d(a,b,3,2,1),nn.ReLU()) for a,b in [(3,16),(16,32),(32,64)]])
        self.cls=nn.ModuleList([nn.Conv2d(c,anchors*classes,3,1,1) for c in [16,32,64]])
        self.box=nn.ModuleList([nn.Conv2d(c,anchors*4,3,1,1) for c in [16,32,64]])
    def forward(self,x):
        outputs=[]
        for block,cls,box in zip(self.blocks,self.cls,self.box):x=block(x);outputs.append((cls(x),box(x)))
        return outputs
class TinyYOLO(nn.Module):
    def __init__(self,classes=20,anchors=3):
        super().__init__();self.net=nn.Sequential(nn.Conv2d(3,32,3,2,1),nn.ReLU(),nn.AdaptiveAvgPool2d((13,13)),nn.Conv2d(32,anchors*(5+classes),1))
    def forward(self,x):return self.net(x)
x=torch.rand(2,3,64,64)
for c,b in TinySSD()(x):print('SSD class/box:',tuple(c.shape),tuple(b.shape))
out=TinyYOLO()(x);print('YOLO shape:',tuple(out.shape));assert out.shape==(2,75,13,13)

## Step 3 — Prepare the labelled dataset and replace the SSD head
A real detector additionally needs anchor matching, losses, decoding and NMS. We use Torchvision SSD for those operations and preserve its pretrained VGG features. There are two classes including background.

In [ ]:
from tutorial_support.detection import *
from torchvision.models.detection import ssd300_vgg16,SSD300_VGG16_Weights
from torchvision.models.detection.ssd import SSDClassificationHead
import pandas as pd
DEVICE=torch_setup();dataset=PennFudan();train,val,test=loaders(dataset)
ssd=ssd300_vgg16(weights=SSD300_VGG16_Weights.DEFAULT)
with torch.no_grad():features=ssd.backbone(torch.zeros(1,3,300,300))
channels=[v.shape[1] for v in features.values()]
ssd.head.classification_head=SSDClassificationHead(channels,ssd.anchor_generator.num_anchors_per_location(),2)
print('SSD feature channels:',channels)
ssd_history=train_detector(ssd,train,DEVICE,1 if QUICK else 10)
ssd_val=evaluate_detector(ssd,val,DEVICE);ssd_test=evaluate_detector(ssd,test,DEVICE)
print('SSD validation:',ssd_val,'test:',ssd_test)
save_result('09_ssd',{'history':ssd_history,'validation':ssd_val,'test':ssd_test})

## Step 4 — Convert boxes into YOLO labels
Torchvision uses absolute xyxy boxes and foreground label 1. YOLO uses zero-based class IDs and normalized centre-x centre-y width height. The converter writes separate train/validation/test manifests; no dataset images are uploaded to GitHub.

In [ ]:
config=export_yolo(dataset);print(config.read_text())
label_path=next((DATA/'penn_yolo'/'labels'/'train').glob('*.txt'))
print('Example annotation:',label_path.read_text())
for line in label_path.read_text().splitlines():
    values=list(map(float,line.split()));assert values[0]==0 and all(0<=v<=1 for v in values[1:])

## Step 5 — Train a pretrained YOLO model and evaluate the held-out split
The small YOLOv8n model downloads official pretrained weights. This is a real training call, not just a dummy forward pass. workers=0 supports Windows/notebook execution. Quick mode is a one-epoch smoke experiment, not convergence.

In [ ]:
from ultralytics import YOLO
import gc
ssd.to('cpu');del ssd;gc.collect()
yolo=YOLO('yolov8n.pt')
yolo.train(data=str(config),epochs=1 if QUICK else 20,imgsz=320,batch=4,workers=0,device=0 if torch.cuda.is_available() else 'cpu',seed=SEED,deterministic=True,project=str(OUT/'yolo'),name='penn_quick' if QUICK else 'penn_full',exist_ok=True,plots=True,verbose=False)
metrics=yolo.val(data=str(config),split='test',imgsz=320,batch=4,workers=0,device=0 if torch.cuda.is_available() else 'cpu',plots=False)
yolo_result={'map':float(metrics.box.map),'map_50':float(metrics.box.map50),'map_75':float(metrics.box.map75)}
display(pd.DataFrame([{'model':'SSD',**ssd_test},{'model':'YOLOv8n',**yolo_result}]))
save_result('09_yolo',yolo_result)

## Step 6 — Display a prediction and discuss limitations
Both models use the same fixed split, but their initialization, resize rules and evaluation implementations differ. Treat this as an instructional comparison. A credible benchmark needs more test examples, matched compute budgets and repeated seeds. The custom TinySSD/TinyYOLO heads above need matching/decoding/loss code before they can be trained as full detectors.

In [ ]:
from PIL import Image
ids=split_ids(dataset)[2]
pred=yolo.predict(str(dataset.images[ids[0]]),conf=.25,verbose=False)[0]
display(Image.fromarray(pred.plot()[...,::-1]))

## References and licensing
[Ultralytics train mode](https://docs.ultralytics.com/modes/train/) and [Torchvision SSD](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.ssd300_vgg16.html). Ultralytics is distributed under AGPL-3.0 with an enterprise option. This repository does not redistribute its weights or third-party datasets.

## Explain it in class
Describe the input, target, model, loss, update rule and evaluation split. Use the displayed results from your own run. Changing the dataset, seed or training budget can change the result.

**Course source:** the matching tutorial PDF supplied in the Deep Learning course. Original explanatory implementation; the course PDFs are not redistributed.